In [0]:
%python
my_schema="""
orderid INT,
customer_id INT,
orderdate DATE,
amount DOUBLE
"""


**STREAMING QUERY**

TO TEST SCHEMA WORKS FINE WE FIRST RUN IN BATCH MODE

In [0]:
%python
df_batch=spark.read.format("csv")\
.option("header","true")\
.schema(my_schema)\
.load("/Volumes/pyspark_cata/source/db_volume/streamSource/")\
.orderBy("orderid")
display(df_batch)

orderid,customer_id,orderdate,amount
1,101,2026-08-15,125.5
2,102,2026-08-16,89.99
3,103,2026-08-17,210.0
4,104,2026-08-18,45.75
5,105,2026-08-19,159.25
6,106,2026-08-20,75.5
7,107,2026-08-21,320.0
8,108,2026-08-22,110.25
9,109,2026-08-23,185.75
10,110,2026-08-24,95.99


In [0]:
%python
df=spark.readStream.format("csv")\
    .option("header","true")\
    .schema(my_schema)\
    .load("/Volumes/pyspark_cata/source/db_volume/streamSource/")

**Querying Output**


In [0]:
%python
df.writeStream \
    .format("delta") \
    .option("checkpointLocation","/Volumes/pyspark_cata/source/db_volume/streamSink/checkpoint") \
    .option("mergeSchema", True) \
    .trigger(once=True) \
    .start("/Volumes/pyspark_cata/source/db_volume/streamSink/data")\
    .awaitTermination()

In [0]:
SELECT *
FROM delta.`/Volumes/pyspark_cata/source/db_volume/streamSink/data/`;

orderid,customer_id,orderdate,amount
1,101,2026-08-15,125.5
2,102,2026-08-16,89.99
3,103,2026-08-17,210.0
4,104,2026-08-18,45.75
5,105,2026-08-19,159.25


In [0]:
%python
display(dbutils.fs.ls("/Volumes/pyspark_cata/source/db_volume/streamSource/"))

path,name,size,modificationTime
dbfs:/Volumes/pyspark_cata/source/db_volume/streamSource/incremental_orders.csv,incremental_orders.csv,164,1788014573000
dbfs:/Volumes/pyspark_cata/source/db_volume/streamSource/initial_orders.csv,initial_orders.csv,153,1788013826000


In [0]:
%python
display(
    dbutils.fs.ls(
        "/Volumes/pyspark_cata/source/db_volume/streamSink/data"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/pyspark_cata/source/db_volume/streamSink/data/_delta_log/,_delta_log/,0,1788015166381
dbfs:/Volumes/pyspark_cata/source/db_volume/streamSink/data/part-00000-04102304-01eb-4f9f-991a-5a1e9d27738a.c000.snappy.parquet,part-00000-04102304-01eb-4f9f-991a-5a1e9d27738a.c000.snappy.parquet,1492,1788013856000


In [0]:
%python
query = df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/pyspark_cata/source/db_volume/streamSink/checkpoint") \
    .option("mergeSchema", True) \
    .trigger(once=True) \
    .start("/Volumes/pyspark_cata/source/db_volume/streamSink/data")

query.awaitTermination()

print(query.lastProgress)

{
    "id": "cbc342ce-46cd-44a8-a0ee-4b1336e93fb0",
    "runId": "e430cc8d-e02c-40fa-9efa-b638e9924994",
    "name": null,
    "timestamp": "2026-08-30T07:04:43.302Z",
    "batchId": 2,
    "batchDuration": 346,
    "durationMs": {
        "triggerExecution": 346,
        "latestOffset": 346
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "FileStreamSource[dbfs:/Volumes/pyspark_cata/source/db_volume/streamSource]",
            "startOffset": "{\"logOffset\":1}",
            "endOffset": "{\"logOffset\":1}",
            "latestOffset": null,
            "numInputRows": 0,
            "inputRowsPerSecond": 0.0,
            "processedRowsPerSecond": 0.0,
            "metrics": {}
        }
    ],
    "sink": {
        "description": "DeltaSink[/Volumes/pyspark_cata/source/db_volume/streamSink/data]",
        "numOutputRows": -1,
        "metrics": {}
    },
    "observedMetrics": {},
    "rtmMetrics": null
}


In [0]:
%python
dbutils.fs.rm(
    "/Volumes/pyspark_cata/source/db_volume/streamSink/checkpoint",
    True
)

True

In [0]:
%python
dbutils.fs.rm(
    "/Volumes/pyspark_cata/source/db_volume/streamSink/data",
    True
)

True

In [0]:
%python
df = spark.readStream.format("csv") \
    .option("header", "true") \
    .schema(my_schema) \
    .load("/Volumes/pyspark_cata/source/db_volume/streamSource/")

In [0]:
%python
query = df.writeStream \
    .format("delta") \
    .option(
        "checkpointLocation",
        "/Volumes/pyspark_cata/source/db_volume/streamSink/checkpoint"
    ) \
    .option("mergeSchema", True) \
    .trigger(availableNow=True) \
    .start("/Volumes/pyspark_cata/source/db_volume/streamSink/data")

query.awaitTermination()

In [0]:
SELECT *
FROM delta.`/Volumes/pyspark_cata/source/db_volume/streamSink/data`
ORDER BY orderid;

orderid,customer_id,orderdate,amount
1,101,2026-08-15,125.5
2,102,2026-08-16,89.99
3,103,2026-08-17,210.0
4,104,2026-08-18,45.75
5,105,2026-08-19,159.25
6,106,2026-08-20,75.5
7,107,2026-08-21,320.0
8,108,2026-08-22,110.25
9,109,2026-08-23,185.75
10,110,2026-08-24,95.99
